# Day 4 · Notebook 3 — Action Groups: Lambda *and* Return-Control

In notebook 2 you wired tools into a **hand-built** loop. Here you give the **managed** agent real capabilities the same way the console does — with an **action group** — and you do it two ways:

- **Part A — Lambda executor.** Bedrock invokes a Lambda you own when the agent picks a tool. The managed twin of notebook 2's tools.
- **Part B — Return-control.** No Lambda. The agent hands the call back to **your code**, you run it in your own process, and you return the result. This is the "add an action group **and use it in code**" path.
- **Part C** — reuse the *same* functions across all three front-ends.

Everything from scratch: the Lambda code, its IAM role, the function, the resource permission, the action group, and `prepare_agent`. **us-east-1** throughout.

> You need broader IAM here: `iam:CreateRole`/`AttachRolePolicy`, `lambda:CreateFunction`/`AddPermission`, and `bedrock-agent:CreateAgentActionGroup`/`PrepareAgent`. If your classroom user lacks these, read the cells — they document exactly what the console does for you.

In [1]:
# Colab: uncomment.  VS Code venv: skip.
# !pip install -q boto3
import boto3, botocore, json, io, zipfile, time, uuid

REGION   = "us-east-1"
AGENT_ID = "3KCNKSOI8U"     # <-- your Agent ID (Bedrock console > Agents > overview)
ALIAS_ID = "TSTALIASID"     # test alias

sts = boto3.client("sts", region_name=REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]
AGENT_ARN  = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:agent/{AGENT_ID}"

iam            = boto3.client("iam")                                   # global service
lam            = boto3.client("lambda",               region_name=REGION)
agent_build    = boto3.client("bedrock-agent",        region_name=REGION)  # CONTROL plane
agent_runtime  = boto3.client("bedrock-agent-runtime", region_name=REGION)  # DATA plane
print("account", ACCOUNT_ID, "| agent", AGENT_ID)

account 123456789012 | agent 3KCNKSOI8U


## Part A — Action group with a Lambda executor

### A1. The Lambda handler — the action-group contract

When the agent calls a function, Bedrock invokes your Lambda with an **event** containing:
`actionGroup`, `function`, and `parameters` (a list of `{name, type, value}`), plus `sessionId` and attributes.

Your Lambda **must** return this exact shape (function-schema form):

```
{ "messageVersion": "1.0",
  "response": { "actionGroup": ..., "function": ...,
    "functionResponse": { "responseBody": { "TEXT": { "body": "<string>" } } } } }
```

`responseBody` only supports **`TEXT`**, and `body` is a string (JSON-encoded is fine). You may add `"responseState": "REPROMPT"` (ask the model to try again on bad input) or `"FAILURE"` (raise a dependency error).

In [2]:
LAMBDA_SRC = r'''
import json

BOOKINGS = {
    "ABC123": {"pnr": "ABC123", "passenger": "R. Mehta", "flight": "6E-203",
               "origin": "BLR", "dest": "DEL", "status": "DISRUPTED", "disruption": "fog at DEL"},
    "ZZ999":  {"pnr": "ZZ999", "passenger": "S. Iyer", "flight": "6E-512",
               "origin": "BOM", "dest": "GOI", "status": "ON_TIME", "disruption": None},
}

def _lookup(pnr):
    b = BOOKINGS.get((pnr or "").upper())
    if not b:
        return {"found": False, "pnr": pnr}
    return {"found": True, **{k: b[k] for k in ("pnr", "passenger", "flight", "origin", "dest", "status")}}

def _disruption(pnr):
    b = BOOKINGS.get((pnr or "").upper())
    if not b:
        return {"found": False, "pnr": pnr}
    return {"pnr": b["pnr"], "status": b["status"], "reason": b["disruption"] or "none"}

def _rebooking(pnr):
    return {"pnr": (pnr or "").upper(), "options": [
        {"flight": "6E-415", "dep": "18:40", "seats": 12},
        {"flight": "6E-422", "dep": "21:10", "seats": 5},
    ]}

DISPATCH = {"lookup_booking": _lookup, "get_disruption_reason": _disruption, "get_rebooking_options": _rebooking}

def lambda_handler(event, context):
    function = event.get("function")
    params = {p["name"]: p.get("value") for p in event.get("parameters", [])}
    fn = DISPATCH.get(function)
    if fn is None:
        body, state = json.dumps({"error": "unknown function: " + str(function)}), "FAILURE"
    else:
        body, state = json.dumps(fn(**params)), None
    fr = {"responseBody": {"TEXT": {"body": body}}}
    if state:
        fr["responseState"] = state
    return {
        "messageVersion": "1.0",
        "response": {"actionGroup": event.get("actionGroup"), "function": function, "functionResponse": fr},
        "sessionAttributes": event.get("sessionAttributes", {}),
        "promptSessionAttributes": event.get("promptSessionAttributes", {}),
    }
'''
print("handler source ready:", len(LAMBDA_SRC), "chars")

handler source ready: 1886 chars


### A2. IAM execution role for the Lambda (from scratch)

A Lambda needs an execution role it can assume (trust `lambda.amazonaws.com`) plus permission to write logs (`AWSLambdaBasicExecutionRole`). The function below is **idempotent** — safe to re-run.

In [3]:
LAMBDA_ROLE_NAME = "travelmind-lambda-role"

def ensure_lambda_role():
    trust = {"Version": "2012-10-17", "Statement": [
        {"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
    try:
        r = iam.create_role(RoleName=LAMBDA_ROLE_NAME,
                            AssumeRolePolicyDocument=json.dumps(trust),
                            Description="TravelMind action-group Lambda execution role")
        arn = r["Role"]["Arn"]
        iam.attach_role_policy(RoleName=LAMBDA_ROLE_NAME,
            PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
        print("created role; waiting ~10s for IAM propagation ...")
        time.sleep(10)
    except iam.exceptions.EntityAlreadyExistsException:
        arn = iam.get_role(RoleName=LAMBDA_ROLE_NAME)["Role"]["Arn"]
        print("role already exists")
    return arn

ROLE_ARN = ensure_lambda_role()
print("role:", ROLE_ARN)

created role; waiting ~10s for IAM propagation ...
role: arn:aws:iam::123456789012:role/travelmind-lambda-role


### A3. Package and create the Lambda function

We zip the handler **in memory** (no files on disk) and create the function. Idempotent: if it exists we just push new code.

In [4]:
FUNCTION_NAME = "travelmind-actions"

def ensure_lambda(role_arn):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("index.py", LAMBDA_SRC)
    code = buf.getvalue()
    try:
        r = lam.create_function(
            FunctionName=FUNCTION_NAME, Runtime="python3.12", Role=role_arn,
            Handler="index.lambda_handler", Code={"ZipFile": code},
            Timeout=15, MemorySize=256, Description="TravelMind action group")
        arn = r["FunctionArn"]
        print("created lambda")
    except lam.exceptions.ResourceConflictException:
        lam.update_function_code(FunctionName=FUNCTION_NAME, ZipFile=code)
        arn = lam.get_function(FunctionName=FUNCTION_NAME)["Configuration"]["FunctionArn"]
        print("lambda exists; pushed new code")
    return arn

FUNCTION_ARN = ensure_lambda(ROLE_ARN)
print("lambda:", FUNCTION_ARN)

lambda exists; pushed new code
lambda: arn:aws:lambda:us-east-1:123456789012:function:travelmind-actions


### A4. Let Bedrock invoke your Lambda — resource-based permission

The agent cannot call your function until you attach a **resource-based policy** allowing the Bedrock principal, scoped to your agent's ARN. Forget this and the action silently fails at invoke time.

In [5]:
def allow_bedrock_invoke():
    try:
        lam.add_permission(
            FunctionName=FUNCTION_NAME,
            StatementId="bedrock-agent-invoke",
            Action="lambda:InvokeFunction",
            Principal="bedrock.amazonaws.com",
            SourceArn=AGENT_ARN,          # scope to THIS agent only (least privilege)
        )
        print("permission added")
    except lam.exceptions.ResourceConflictException:
        print("permission already present")

allow_bedrock_invoke()

permission added


### A5. Create the action group + `prepare_agent`

`functionSchema` is the code-first way to define tools (simpler than an OpenAPI schema). `actionGroupExecutor={"lambda": ARN}` points the agent at your function. Then **`prepare_agent`** — every change to a managed agent needs it before it takes effect (the API twin of the *Prepare* button).

In [6]:
FUNCTION_SCHEMA = {"functions": [
    {"name": "lookup_booking",
     "description": "Look up a booking by PNR. Returns passenger, flight, route, and status.",
     "parameters": {"pnr": {"description": "6-character PNR", "type": "string", "required": True}}},
    {"name": "get_disruption_reason",
     "description": "Given a PNR, return why the flight was disrupted, or 'none'.",
     "parameters": {"pnr": {"type": "string", "required": True}}},
    {"name": "get_rebooking_options",
     "description": "Given a PNR, return alternative flights the passenger can move to.",
     "parameters": {"pnr": {"type": "string", "required": True}}},
]}

def ensure_action_group():
    try:
        agent_build.create_agent_action_group(
            agentId=AGENT_ID, agentVersion="DRAFT",
            actionGroupName="travelmind-bookings",
            actionGroupExecutor={"lambda": FUNCTION_ARN},
            functionSchema=FUNCTION_SCHEMA,
            actionGroupState="ENABLED",
            description="Booking lookup / disruption / rebooking, executed by Lambda")
        print("action group created")
    except agent_build.exceptions.ConflictException:
        print("action group already exists (use update_agent_action_group to change it)")

ensure_action_group()
agent_build.prepare_agent(agentId=AGENT_ID)
print("prepare_agent requested; wait ~20-30s before invoking")
time.sleep(25)

action group created
prepare_agent requested; wait ~20-30s before invoking


### A6. Use it — the agent now calls your Lambda

Invoke as usual. In the trace you will see the agent pick `get_rebooking_options`, Bedrock invoke your Lambda, and the result flow back into the answer.

In [15]:
def ask_agent(prompt, session_id=None, trace=True):
    session_id = session_id or str(uuid.uuid4())
    resp = agent_runtime.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID,
        sessionId=session_id, inputText=prompt, enableTrace=trace)
    answer, traces = "", []
    for event in resp["completion"]:
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode("utf-8")
        elif "trace" in event:
            traces.append(event["trace"]["trace"])
    return answer, traces, session_id

def ask_agent_safe(prompt, tries=4, base=8, **kw):
    for i in range(tries):
        try:
            return ask_agent(prompt, **kw)
        except (botocore.exceptions.EventStreamError, botocore.exceptions.ClientError) as e:
            msg = str(e).lower()
            transient = any(k in msg for k in ("dependencyfailed", "timeout", "throttl", "serviceunavailable"))
            if i == tries - 1 or not transient:
                raise
            wait = base * (2 ** i)
            print(f"retry {i+1}/{tries} in {wait}s: {str(e)[:90]}")
            time.sleep(wait)

answer, traces, _ = ask_agent_safe("My flight ABC123 was disrupted. Why?")
for t in traces:
    ot = t.get("orchestrationTrace", {})
    if "invocationInput" in ot:
        print("ACT  :", json.dumps(ot["invocationInput"])[:240])
    if ot.get("observation", {}).get("finalResponse"):
        print("FINAL:", ot["observation"]["finalResponse"]["text"][:300])
print("\nANSWER:", answer)

## If you want to see the raw Lambda logs, uncomment below. Note that if the Lambda was never invoked, there won't be any log stream or log group, so both cases are handled.
# logs  = boto3.client("logs", region_name=REGION)
# group = "/aws/lambda/travelmind-actions"

# try:
#     s = logs.describe_log_streams(logGroupName=group, orderBy="LastEventTime",
#                                   descending=True, limit=1)["logStreams"]
#     if not s:
#         print("NO log stream -> Lambda was never invoked. Failure is upstream (model/prepare/throttle).")
#     else:
#         for e in logs.get_log_events(logGroupName=group, logStreamName=s[0]["logStreamName"],
#                                      limit=40, startFromHead=False)["events"]:
#             print(e["message"].rstrip())
# except logs.exceptions.ResourceNotFoundException:
#     print("NO log group -> Lambda never invoked. Failure is upstream.")

ACT  : {"actionGroupInvocationInput": {"actionGroupName": "BookingActions", "executionType": "LAMBDA", "function": "lookup_booking", "parameters": [{"name": "pnr", "type": "string", "value": "ABC123"}]}, "invocationType": "ACTION_GROUP", "traceId"
ACT  : {"actionGroupInvocationInput": {"actionGroupName": "BookingActions", "executionType": "LAMBDA", "function": "get_disruption_reason", "parameters": [{"name": "pnr", "type": "string", "value": "ABC123"}]}, "invocationType": "ACTION_GROUP", "t
FINAL: The flight ABC123 was disrupted due to heavy fog at the origin.

ANSWER: The flight ABC123 was disrupted due to heavy fog at the origin.


## Part B — Drive an action group from your own code (return-control)

A Lambda is not the only executor. Set the executor to **`RETURN_CONTROL`** and Bedrock stops calling a Lambda — instead `invoke_agent` hands the predicted call back to you in a **`returnControl`** event. You run the function **in your own process**, then send the result back through `sessionState`.

Why you would want this:
- run tool logic **inside your service / VPC** — sensitive data never leaves your perimeter
- **no Lambda cold starts**, no extra deploy artifact
- reuse code you already have (the exact functions from notebook 2)

We add a separate action group `code-actions` with one function, `hold_seat`, so it does not clash with the Lambda group above.

In [16]:
RC_SCHEMA = {"functions": [
    {"name": "hold_seat",
     "description": "Place a temporary hold on a seat for a PNR on a given flight. Returns a hold id.",
     "parameters": {"pnr": {"type": "string", "required": True},
                    "flight": {"type": "string", "required": True}}},
]}

def ensure_rc_action_group():
    try:
        agent_build.create_agent_action_group(
            agentId=AGENT_ID, agentVersion="DRAFT",
            actionGroupName="code-actions",
            actionGroupExecutor={"customControl": "RETURN_CONTROL"},   # <-- the key line
            functionSchema=RC_SCHEMA,
            actionGroupState="ENABLED",
            description="Executed by the application via return-control")
        print("return-control action group created")
    except agent_build.exceptions.ConflictException:
        print("return-control action group already exists")

ensure_rc_action_group()
agent_build.prepare_agent(agentId=AGENT_ID)
time.sleep(25)
print("ready")

return-control action group created
ready


### B1. The driver — run the function in your code, send the result back

Two invocations:
1. First `invoke_agent` returns a **`returnControl`** event with `invocationId` and the predicted `function` + `parameters`.
2. You execute the function, then call `invoke_agent` again with `sessionState.returnControlInvocationResults`. The `invocationId` must match. (`inputText` is ignored on the second call.)

In [18]:
def hold_seat(pnr, flight):
    # your real code — runs in your process, not a Lambda
    return {"hold_id": f"H-{pnr.upper()}-{flight}", "status": "HELD", "expires_min": 20}

CODE_TOOLS = {"hold_seat": hold_seat}

def run_with_return_control(prompt, session_id=None):
    session_id = session_id or str(uuid.uuid4())
    resp = agent_runtime.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID,
        sessionId=session_id, inputText=prompt, enableTrace=False)

    rc, answer = None, ""
    for event in resp["completion"]:
        if "chunk" in event:
            answer += event["chunk"]["bytes"].decode("utf-8")
        elif "returnControl" in event:
            rc = event["returnControl"]

    if rc is None:
        return answer, session_id                      # agent answered without your code

    inv_id, results = rc["invocationId"], []
    for inv in rc["invocationInputs"]:
        fi = inv["functionInvocationInput"]
        args = {p["name"]: p.get("value") for p in fi.get("parameters", [])}
        print(f"agent -> your code: {fi['function']}({args})")
        out = CODE_TOOLS[fi["function"]](**args)        # run it locally
        results.append({"functionResult": {
            "actionGroup": fi["actionGroup"], "function": fi["function"],
            "responseBody": {"TEXT": {"body": json.dumps(out)}}}})

    resp2 = agent_runtime.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID,
        sessionId=session_id, enableTrace=False,        # inputText omitted on purpose
        sessionState={"invocationId": inv_id, "returnControlInvocationResults": results})
    final = ""
    for event in resp2["completion"]:
        if "chunk" in event:
            final += event["chunk"]["bytes"].decode("utf-8")
    return final, session_id

final, _ = run_with_return_control("Hold a seat for PNR ABC123 on flight 6E-415.")
print("\nANSWER:", final)


ANSWER: Seat hold request submitted for PNR ABC123 on flight 6E-415.


## Part C — One registry, three front-ends

The booking functions are the same logic whether they run as a **Lambda action group** (Part A), via **return-control** in your code (Part B), or inside the **hand-built Converse loop** (notebook 2). Write the business logic once; choose the front-end per requirement.

In [19]:
# the same callables, reused. (lookup/disruption/rebooking from nb2; hold_seat from Part B)
REGISTRY = {
    "lookup_booking":        "Part A (Lambda) and nb2 loop",
    "get_disruption_reason": "Part A (Lambda) and nb2 loop",
    "get_rebooking_options": "Part A (Lambda) and nb2 loop",
    "hold_seat":             "Part B (return-control), runs in your process",
}
for fn, where in REGISTRY.items():
    print(f"{fn:24s} -> {where}")

lookup_booking           -> Part A (Lambda) and nb2 loop
get_disruption_reason    -> Part A (Lambda) and nb2 loop
get_rebooking_options    -> Part A (Lambda) and nb2 loop
hold_seat                -> Part B (return-control), runs in your process


## Lambda vs return-control vs hand-built — when to use which

| | Lambda action group | Return-control | Hand-built Converse loop |
|---|---|---|---|
| Who runs the tool | AWS invokes your Lambda | **Your app** | Your app |
| Loop owner | Bedrock (managed) | Bedrock (managed) | **You** |
| Data locality | leaves to Lambda | **stays in your process / VPC** | stays in your process |
| Cold starts / deploy | yes | none | none |
| Best for | clean managed tools, async scale | sensitive data, reuse existing code, human-in-the-loop | full control, debugging, custom routing |

**Production notes:** scope the Lambda permission to the **agent ARN** (done above), give the Lambda role **least privilege** (only what the tool needs), keep infra creation **idempotent**, and run **`prepare_agent` in CI** after any change so deploys are repeatable.

## Recap

- An **action group** gives the managed agent tools. Two executors: **Lambda** (AWS runs it) or **RETURN_CONTROL** (your code runs it).
- From scratch you built: handler → IAM role → function → resource permission → action group → `prepare_agent`.
- **Return-control** is how you "use the action group in code" — the agent decides, your process executes, you send results back via `sessionState`.
- Same functions, three front-ends.

**Next — Notebook 4:** control the agent's behavior — stop the runaway loop (spin-detection + a `handoff_to_human` fallback tool), tune inference (and when not to), and block hallucinations with a contextual grounding guardrail.